# L6-3d Demo: Domain randomization and robustness

This notebook trains:
- a nominal policy on the default crawler
- a DR policy whose mass, friction, and perturbations vary episode to episode

The lecture framing is the standard sim-to-real one: if the inner loop only sees one nominal simulator, the outer objective is overfit to that simulator. Domain randomization broadens the training distribution and should give up some nominal specialization in exchange for perturbation robustness.

Lecture citation anchors: `vuong2019dr`, `jakobi1997minimal`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from l6_3_utils import (
    CMM_AMBER,
    CMM_ROSE,
    CMM_SLATE,
    FIGURE_DIR,
    compare_policy_push_robustness,
    save_figure,
    train_or_load,
)

TOTAL_TIMESTEPS = 50_000
SEED = 17
ENV_KWARGS = dict(
    include_velocity=True,
    action_mode="pd_target",
    reward_mode="dense_vel",
)
DR_CONFIG = dict(
    mass_range=(0.7, 1.3),
    friction_range=(0.7, 1.3),
    push_prob=0.3,
    push_mag=8.0,
)

print(f"Saving lecture figures into: {FIGURE_DIR}")


In [ ]:
nominal_artifact = train_or_load(
    "ppo",
    "l6_3d_ppo_nominal.zip",
    ENV_KWARGS,
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
)
dr_artifact = train_or_load(
    "ppo",
    "l6_3d_ppo_dr.zip",
    ENV_KWARGS,
    total_timesteps=TOTAL_TIMESTEPS,
    seed=SEED,
    dr_config=DR_CONFIG,
)


In [ ]:
push_magnitudes = np.arange(0, 22, 2, dtype=float)
results = compare_policy_push_robustness(
    {
        "nominal": nominal_artifact.model,
        "trained with DR": dr_artifact.model,
    },
    ENV_KWARGS,
    push_magnitudes,
    n_trials=10,
    push_step=75,
    seed=SEED,
)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), sharex=True)
style = {
    "nominal": dict(color=CMM_ROSE, marker="o"),
    "trained with DR": dict(color=CMM_AMBER, marker="s"),
}

for label, result in results.items():
    axes[0].plot(
        result["push_magnitudes"],
        result["survival_fraction"],
        lw=2.0,
        label=label,
        **style[label],
    )
    axes[1].plot(
        result["push_magnitudes"],
        result["mean_displacement"],
        lw=2.0,
        label=label,
        **style[label],
    )

axes[0].set_xlabel("push magnitude (N)")
axes[0].set_ylabel("survival fraction")
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title("Survival under a late push", color=CMM_SLATE)
axes[0].grid(True, alpha=0.25)
axes[0].legend(loc="upper right", fontsize=8)

axes[1].set_xlabel("push magnitude (N)")
axes[1].set_ylabel("mean final x-position")
axes[1].set_title("Forward progress after the push", color=CMM_SLATE)
axes[1].grid(True, alpha=0.25)

fig_path = save_figure(fig, "dr_push_survival.pdf")
fig_path


Classroom live-demo command with the DR-trained cached policy:

```bash
uv run python teleop_crawler.py --policy saved_checkpoints/l6_3d_ppo_dr.zip --policy-algo ppo --action-mode pd_target
```

Assignment 3 uses the same knobs at higher fidelity on the Pi biped: `randomize_friction`, `randomize_base_mass`, `push_robots`, and `dynamic_randomization`.
